### Step 1: Setup & Connect to Database

In [1]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
import warnings

# Ignore warning messages for a cleaner output
warnings.filterwarnings('ignore')

# 1. Connect to the SQLite database
db_path = '../data/db/bluestock_mf.db'
conn = sqlite3.connect(db_path)
print("successfully connected to the database")

# 2. Load DataFrames using SQL Queries
# Load Dimension Table
df_fund = pd.read_sql_query("SELECT * FROM dim_fund", conn)

# Load Fact Tables
df_nav = pd.read_sql_query("SELECT * FROM fact_nav", conn)
df_tx = pd.read_sql_query("SELECT * FROM fact_transactions", conn)
df_aum = pd.read_sql_query("SELECT * FROM fact_aum", conn)
df_perf = pd.read_sql_query("SELECT * FROM fact_performance", conn)

# Print success message and data shapes
print("\n All Data loaded successfully into Pandas DataFrames!")
print("-" * 50)
print(f"Fund Details: {df_fund.shape[0]} rows")
print(f"Nav History: {df_nav.shape[0]} rows")
print(f"Transactions: {df_tx.shape[0]} rows")
print(f"AUM Data: {df_aum.shape[0]} rows")
print(f"Performance: {df_perf.shape[0]} rows")

successfully connected to the database

 All Data loaded successfully into Pandas DataFrames!
--------------------------------------------------
Fund Details: 40 rows
Nav History: 128640 rows
Transactions: 65556 rows
AUM Data: 90 rows
Performance: 40 rows


### STEP 2: TIME-SERIES ANALYSIS

In [2]:
# 1. NAV Trend Analysis (2022-2026)
# Convert nav_date to datetime
df_nav['nav_date'] = pd.to_datetime(df_nav['nav_date'])

# Merge NAV data with Fund Dimension to get scheme names
df_nav_merged = pd.merge(df_nav, df_fund[['amfi_code','scheme_name']], on='amfi_code')

# Plotly Line Chart for NAV
fig1 = px.line(df_nav_merged, x='nav_date', y='nav', color='scheme_name',
               title='1. Daily NAV Trend for 40 Schemes (2022-2026)',
               labels={'nav_date': 'Date', 'nav': 'Net Asset Value (INR)'})

# Highlight 2023 Bull Run (Green Zone)
fig1.add_vrect(x0="2023-04-01", x1="2023-12-31", fillcolor="green", opacity=0.1, 
               line_width=0, annotation_text="2023 Bull Run", annotation_position="top left")

# Highlight 2024 Market Correction (Red Zone)
fig1.add_vrect(x0="2024-05-01", x1="2024-07-31", fillcolor="red", opacity=0.1, 
               line_width=0, annotation_text="2024 Correction", annotation_position="top left")

# Hide legend to make the chart clean (since 40 funds will make it crowded)
fig1.update_layout(showlegend=False)
fig1.show()


# 2. SIP Inflow Time-Series (Jan 2022 - Dec 2025)
# Convert transaction_date to datetime
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

# Filter only SIPs and create Year-Month column
sip_data = df_tx[df_tx['transaction_type'] == 'SIP'].copy()
sip_data['month_year'] = sip_data['transaction_date'].dt.to_period('M').astype(str)

# Group by month
monthly_sip = sip_data.groupby('month_year')['amount_inr'].sum().reset_index()

fig2 = px.bar(monthly_sip, x='month_year', y='amount_inr',
              title='2. Monthly SIP Inflow Trend',
              labels={'month_year': 'Month-Year', 'amount_inr': 'Total SIP Amount (INR)'})

# Annotate the All-Time High in Dec 2025
max_idx = monthly_sip['amount_inr'].idxmax()
max_month = monthly_sip.loc[max_idx, 'month_year']
max_value = monthly_sip.loc[max_idx, 'amount_inr']

# Annotate the All-Time High dynamically
fig2.add_annotation(x=max_month, y=max_value,
                    text=f"All-time high: ₹{max_value:,.0f}", 
                    showarrow=True, arrowhead=1, ay=-40)
fig2.show()


# 3. Folio Count Growth (Industry Level Milestone)
# Creating synthetic industry data as per project requirement
folio_data = pd.DataFrame({
    'Date': pd.to_datetime(['2022-01-01', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']),
    'Folio_Count_Cr': [13.26, 14.20, 16.50, 21.00, 26.12]
})

# Plotly Line Chart with Markers
fig3 = px.line(folio_data, x='Date', y='Folio_Count_Cr', markers=True,
               title= '3. Mutual Fund Folio Count Growth (Jan 2022 - Dec 2025)')

# Mark Key Milestones
fig3.add_annotation(x='2022-01-01', y=13.26, text="Start: 13.26 Cr", showarrow=True, ay=-30)
fig3.add_annotation(x='2025-12-31', y=26.12, text='End: 26.12 Cr', showarrow=True, ay=30)
fig3.show()

ValueError: You are trying to merge on str and int64 columns for key 'amfi_code'. If you wish to proceed you should use pd.concat

### STEP 3: MARKET SIZE & CATEGORY ANALYSIS

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# 1. AUM Growth Bar Chart (2022-2025)
df_aum['year'] = pd.to_datetime(df_aum['quarter_date']).dt.year

# Get the maximum AUM for each fund house per year
yearly_aum = df_aum.groupby(['year', 'fund_house'])['aum_crore'].max().reset_index()

plt.figure(figsize=(14, 6))
ax = sns.barplot(data=yearly_aum, x='year', y='aum_crore', hue='fund_house', palette='viridis')

plt.title('4. yearly AUM Growth by Fund House (2022-2025)')
plt.xlabel('year')
plt.ylabel('AUM (in Crore INR)')

# Highlight SBI at ~12.5L Cr dominance (Assuming SBI is in the data and reached highest in 2025)
sbi_2025_aum = yearly_aum[(yearly_aum['fund_house'].str.contains('SBI')) & (yearly_aum['year'] == 2025)]['aum_crore'].max()
if pd.notna(sbi_2025_aum):
    plt.annotate('SBI Dominance:\n~12.5L Cr',
                 xy=(3, sbi_2025_aum),
                 xytext=(3, sbi_2025_aum + 50000),
                 arrowprops=dict(facecolor='red', shrink=0.05),
                 fontsize=11, color='red', weight='bold', ha='center')
    
plt.legend(title='Fund House', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


# 2. Category Inflow Heatmap
df_tx_fund = pd.merge(df_tx, df_fund[['amfi_code', 'category']], on='amfi_code')

# Filter for inflows (SIP & Lumpsum) and create Year-Month column
df_inflow = df_tx_fund[df_tx_fund['transaction_type'].isin(['SIP', 'Lumpsum'])].copy()
df_inflow['month_year'] = pd.to_datetime(df_inflow['transaction_date']).dt.to_period('M').astype(str)

# Create a pivot table for the heatmap
heatmap_data = df_inflow.pivot_table(index='category', columns='month_year',values='amount_inr', aggfunc='sum').fillna(0)

plt.figure(figsize=(40, 6))
sns.heatmap(heatmap_data, cmap='YlGnBu', linewidths=.5, annot=False)
plt.title('5. Category inflow heatmap (Net Inflow Intensity)')
plt.xlabel('Mont-Year')
plt.ylabel('Fund Category')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 3. Sector Allocation Donut Chart
# Load the portfolio holdings data from raw folder
try: 
    df_holdings = pd.read_csv('../data/raw/09_portfolio_holdings.csv')
    
    # Merge with fund data to filter ONLY Equity funds
    df_eq_funds = df_fund[df_fund['category'] == 'Equity'][['amfi_code']]
    
    df_holdings['amfi_code'] = df_holdings['amfi_code'].astype(str)
    df_eq_funds['amfi_code'] = df_eq_funds['amfi_code'].astype(str)   
    
    df_eq_holdings = pd.merge(df_holdings, df_eq_funds, on='amfi_code')    
     
    # Aggregate weights by sector
    sector_weights = df_eq_holdings.groupby('sector')['weight_pct'].sum().sort_values(ascending=False)
    
    # Take top 8 sectors, group the rest as 'Others' to make the chart clean
    top_sectors = sector_weights.head(8)
    others = pd.Series([sector_weights[8:].sum()], index=['Others'])
    final_sectors = pd.concat([top_sectors, others])
    
    # Plot Donut Chart
    plt.figure(figsize=(8, 8))
    plt.pie(final_sectors, labels=final_sectors.index, autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('pastel'), wedgeprops=dict(width=0.4, edgecolor='w'))
    
    plt.title('6. Sector Allocation Across All equity Funds')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error loading portfolio holdings: {e}. please ensure '09_portfolio_holdings.csv' exists in the raw folder")
    



### STEP 4: INVESTOR DEMOGRAPHICS

In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# Load the processed transactions dataset to get demographic columns (age, gender, state, etc.)
df_tx_demo = pd.read_csv('../data/processed/clean_transaction.csv')

# 1. Age Group Distribution (Pie Chart)
age_counts =df_tx_demo['age_group'].value_counts().reset_index()
age_counts.columns= ['Age Group', 'Count']

fig_age = px.pie(age_counts, values='Count', names='Age Group',
                 title='7. Investor Age Group Distributions', hole=0.3,
                 color_discrete_sequence=px.colors.sequential.Teal)
fig_age.show()


# 2. Gender Split (Pie Chart)
gender_counts = df_tx_demo['gender'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']

fig_gender = px.pie(gender_counts, values='Count', names='Gender', 
                    title='8. Investor Gender Split',
                    color='Gender', color_discrete_map={'Male': '#636EFA', 'Female': '#EF553B', 'Other': '#00CC96'})
fig_gender.show()


# 3. SIP Amount Box Plot by Age Group
# Filter only SIP transactions
sip_demo = df_tx_demo[df_tx_demo['transaction_type'] == 'SIP']
sns.boxplot(data=sip_demo, x='age_group', y='amount_inr', palette='Set2')
plt.title('9. SIP Amount Distribution by Age Group')
plt.xlabel('Age Group')
plt.ylabel('SIP Amount (INR)')
plt.tight_layout()
plt.show()


# 4. Geographic Distribution (State-wise SIP)
state_sip = sip_demo.groupby('state')['amount_inr'].sum().reset_index().sort_values(by='amount_inr', ascending=False)

plt.figure(figsize=(12,10))
sns.barplot(data=state_sip, x='amount_inr', y='state', palette='Blues_r')
plt.title("10. Total SIP Amount by state")
plt.xlabel('Total SIP Amount (INR)')
plt.ylabel('State')
plt.tight_layout()
plt.show()


# 5. T30 vs B30 City Tier (Pie Chart)
city_tier_counts = df_tx_demo['city_tier'].value_counts().reset_index()
city_tier_counts.columns = ['City Tier', 'Count']

fig_tier = px.pie(city_tier_counts, values='Count', names='City Tier',
                  title= '11. city Tier Distribution (T30: Top 30 Cities, B30:Beyond 30)',
                  hole=.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig_tier.show()


### STEP 5: FUND PERFORMANCE ANALYSIS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. NAV Return Correlation Matrix (Top 10 Funds)
# Step 1: Merge NAV data with Fund Dimension to get scheme names
df_nav_merged = pd.merge(df_nav, df_fund[['amfi_code', 'scheme_name']], on='amfi_code')

# Step 2: Create a Pivot Table with nav_date as index, scheme_name as columns, and nav as values
nav_pivot = df_nav_merged.pivot_table(index='nav_date', columns='scheme_name', values='nav')

# Step 3: Select any 10 funds to keep the heatmap clean and readable
selected_funds = nav_pivot.columns[:10]
nav_pivot_10 = nav_pivot[selected_funds]

# Step 4: Calculate Daily Returns (Percentage Change)
daily_returns = nav_pivot_10.pct_change().dropna()

# Step 5: Compute Correlation Matrix
correlation_matrix = daily_returns.corr()

# Step 6: Plot the Seaborn Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f",
            linewidths=0.5, vmin=-0.5, vmax=1, square=True)

plt.title('12. Daily Return Correlation Matrix (10 Selected Funds)')
plt.xlabel('Mutual Fund Schemes')
plt.ylabel('Mutual Fund Schemes')

# Rotate the x-axis labels so they don't overlap
plt.xticks(rotation=45,ha='right')
plt.tight_layout()
plt.show()


#  Key EDA Findings & Business Insights

Based on the Exploratory Data Analysis (EDA) of the Bluestock Mutual Fund dataset, here are the top 10 business insights:

1. **Massive 2023 Bull Run:** The Daily NAV trend chart clearly highlights a massive market rally throughout 2023 across almost all 40 equity and hybrid schemes, followed by a brief correction phase in mid-2024. *(Ref: Chart 1)*
2. **Record SIP Inflows:** Monthly SIP contributions show a steady upward trajectory, peaking at an all-time high of ₹31,002 Crore in December 2025, indicating strong retail investor confidence. *(Ref: Chart 2)*
3. **Consistent Folio Growth:** The mutual fund industry witnessed a linear and consistent growth in investor accounts (folios), doubling from 13.26 Cr in Jan 2022 to 26.12 Cr in Dec 2025. *(Ref: Chart 3)*
4. **AMC Market Dominance:** SBI Mutual Fund maintains a clear monopoly in Assets Under Management (AUM), dominating the market with approximately ₹12.5 Lakh Crore by 2025. *(Ref: Chart 4)*
5. **Equity Over Debt:** The Category Inflow Heatmap reveals a much higher and consistent net inflow intensity towards Equity funds compared to Debt funds, especially noticeable in the late 2024 - 2025 period. *(Ref: Chart 5)*
6. **Sector Bias (Banking & IT):** Portfolio holdings across equity funds are heavily skewed towards the Banking sector (19.2%), followed by IT (13.4%) and Pharma (12.0%). *(Ref: Chart 6)*
7. **Young Investors Leading:** The 26-35 age demographic forms the largest segment of mutual fund investors (41.1%), highlighting the success of digital KYC and modern investment apps. *(Ref: Chart 7)*
8. **Gender Disparity:** There is a significant gender gap in mutual fund investments, with male investors contributing to roughly 66.5% of the total investor base compared to 33.5% female investors. *(Ref: Chart 8)*
9. **SIP Ticket Size Consistency:** The boxplot distribution shows that while the median SIP amount remains relatively similar across age groups, the 36-45 age group exhibits the highest number of large-value outlier SIPs. *(Ref: Chart 9)*
10. **Urban Concentration (T30):** Geographic analysis confirms that mutual fund penetration is heavily concentrated in Top 30 (T30) cities (~66.3%), with states like Maharashtra, Delhi, and West Bengal bringing in the highest SIP volumes. *(Ref: Chart 10 & 11)*